In [1]:
from __future__ import annotations
import os, json
import random
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Any

import matplotlib.pyplot as plt
import sys
import importlib
from IPython.display import Image

# Load .env 
from dotenv import load_dotenv
load_dotenv()

# Add project root to Python path
project_root = Path.cwd().parent 
sys.path.insert(0, str(project_root))

from src.utils.gpqa_sampler import create_gpqa_quiz
from src.graphs.adaptive_refinement_graph import create_adaptive_refinement_graph, create_initial_state as adaptive_state
from src.graphs.baseline_graph import create_baseline_graph, create_initial_state as baseline_state
from src.agents.pairwise_judge_agent import batch_pairwise_comparison
from src.config.agent_config import _llm, PERSONAS

C:\Users\vedan\Desktop\CS 329T\Homeworks\Homework 1\venv\Lib\site-packages\munch\__init__.py:24: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
# Display model environment variables used by agents
print({k: os.getenv(k) for k in [
    "MODEL_NAME",
    "TEACHER_MODEL",
    "GRADING_MODEL",
    "COORDINATOR_MODEL",
    "STUDENT_MODEL",
    "CRITIQUE_EVAL_MODEL",
]})

{'MODEL_NAME': 'gpt-4o', 'TEACHER_MODEL': 'gpt-4o', 'GRADING_MODEL': 'gpt-4o', 'COORDINATOR_MODEL': 'gpt-54o', 'STUDENT_MODEL': 'gpt-4o', 'CRITIQUE_EVAL_MODEL': 'gpt-4o'}


In [3]:
try:
    from trulens.providers.openai import OpenAI as TruOpenAI
    tru_provider = TruOpenAI(model_engine="gpt-4o-mini")
except Exception as e:
    tru_provider = None
    print("TruLens provider init failed:", e)

In [4]:
"""
Evaluation pipeline comparing zero-shot, baseline, and adaptive explanation systems.

Runs all three approaches on the same quiz questions and saves:
- Quiz performance metrics for each system
- Final explanations for baseline and adaptive (for pairwise judge comparison)
"""

class EvaluationPipeline:
    """Pipeline for comparing zero-shot, baseline, and adaptive systems."""
    
    def __init__(
        self,
        subset: str = "gpqa_main",
        domain: str = "Physics",
        n_questions: int = 10,
        seed: int = 53,
        results_dir: Path = Path("results")
    ):
        self.subset = subset
        self.domain = domain
        self.n_questions = n_questions
        self.seed = seed
        self.results_dir = Path(results_dir)
        self.results_dir.mkdir(parents=True, exist_ok=True)
        
        self.rng = random.Random(seed)
        
    def generate_quiz(self) -> List[Dict[str, Any]]:
        """Generate quiz questions from GPQA dataset."""
        quiz, indices = create_gpqa_quiz(subset = self.subset,
                                         domain = self.domain,
                                         seed = self.seed,
                                         num_questions = self.n_questions)
        return quiz, indices
    
    def run_zero_shot(self, gpqa_question: Dict[str, Any]) -> Dict[str, Any]:
        """
        Run zero-shot: LLM answers quiz directly without explanation.
        
        Returns quiz performance metrics only (no explanation).
        """
        print(f"  Running zero-shot...")
        
        # Direct answer without explanation
        llm = _llm(temperature=0.7, role="zero_shot_answerer")
        
        question = gpqa_question["question"]
        options = gpqa_question["options"]
        correct = gpqa_question["correct"]
        
        prompt = f"""Answer this physics question by selecting the best option.

        Question: {question}
        
        Options:
        {chr(10).join(options)}
        
        Provide ONLY the letter of your answer (A, B, C, or D) and a brief one-sentence explanation.
        
        Format your response as:
        Answer: [LETTER]
        Explanation: [One sentence]"""
        
        response = llm.invoke(prompt).content
        
        # Parse response
        answer_line = [line for line in response.split('\n') if 'Answer:' in line]
        explanation_line = [line for line in response.split('\n') if 'Explanation:' in line]
        
        predicted = answer_line[0].split('Answer:')[-1].strip()[0] if answer_line else "?"
        explanation = explanation_line[0].split('Explanation:')[-1].strip() if explanation_line else ""
        
        is_correct = predicted == correct
        
        return {
            "quiz_results": {
                "total_questions": 1,
                "question_id": gpqa_question["id"],
                "predicted": predicted,
                "is_correct": is_correct,
                "explanation_one_liner": explanation,
            },
            "overall_score": 1.0 if is_correct else 0.0,
            "explanation": None  # No detailed explanation for zero-shot
        }
    
    def run_baseline(
        self, 
        gpqa_question: Dict[str, Any],
        baseline_graph,
    ) -> Dict[str, Any]:
        """
        Run baseline: Single teacher explanation (zero-shot) → grading.
        
        Returns quiz performance metrics and explanation.
        """
        print(f"  Running baseline...")
        
        # Run baseline graph
        baseline_results = baseline_graph.invoke(baseline_state(gpqa_question),
                                                 config={"recursion_limit": 30})

        quiz_results = baseline_results.get("quiz_results", {})
        is_correct = quiz_results.get("is_correct", False)
        
        return {
            "quiz_results": quiz_results,
            "overall_score": 1.0 if is_correct else 0.0,
            "is_correct": is_correct,
            "explanation": baseline_results.get("explanation", ""),
            "single_answer": baseline_results.get("single_answer", ""),
            "single_explanation": baseline_results.get("single_explanation", "")
        }
    
    def run_adaptive(
        self,
        gpqa_question: Dict[str, Any],
        adaptive_graph,
        max_iters: int = 3
    ) -> Dict[str, Any]:
        """
        Run adaptive: Full multi-agent refinement → grading.
        
        Returns quiz performance metrics and final explanation.
        """
        print(f"  Running adaptive (max {max_iters} iterations)...")
        
        # Run adaptive graph
        adaptive_results = adaptive_graph.invoke(adaptive_state(gpqa_question, max_iters = max_iters),
                                                    config={"recursion_limit": 30})

        quiz_results = adaptive_results.get("quiz_results", {})
        is_correct = quiz_results.get("is_correct", False)
        
        return {
            "quiz_results": quiz_results,
            "overall_score": 1.0 if is_correct else 0.0,
            "is_correct": is_correct,
            "explanation": adaptive_results.get("explanation", ""),
            "single_answer": adaptive_results.get("single_answer", ""),
            "single_explanation": adaptive_results.get("single_explanation", ""),
            "iterations": adaptive_results.get("iteration", 0),
            "final_scores": adaptive_results.get("reward_scores", {}),
            "history": adaptive_results.get("history", [])
        }
    
    def run_full_evaluation(
        self, 
        baseline_graph = None,
        adaptive_graph = None,
        max_iters: int = 3
    ) -> Dict[str, Any]:
        """
        Run complete evaluation: zero-shot, baseline, and adaptive on same quiz.
        
        Args:
            baseline_graph: Pre-built baseline graph (or will create if None)
            adaptive_graph: Pre-built adaptive graph (or will create if None)
            max_iters: Max iterations for adaptive refinement
            
        Returns:
            Complete evaluation results with all metrics and explanations
        """
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        print(f"\n{'='*80}")
        print(f"Starting Evaluation Run: {timestamp}")
        print(f"{'='*80}")
        print(f"Dataset: {self.subset} - {self.domain}")
        print(f"Questions: {self.n_questions}")
        print(f"Seed: {self.seed}\n")

        # Create graphs if not provided
        if baseline_graph is None:
            print("Creating baseline graph...")
            baseline_graph = create_baseline_graph()
        
        if adaptive_graph is None:
            print("Creating adaptive graph...")
            adaptive_graph = create_adaptive_refinement_graph()
        
        # Generate quiz
        print("Generating quiz...")
        quiz, indices = self.generate_quiz()
        print(f"Generated {len(quiz)} questions\n")
        
        # Store results for each question
        question_results = []
        
        for i, gpqa_question in enumerate(quiz, 1):

            question = gpqa_question.get("question", "")
            question_id = gpqa_question.get("id", "")
            
            print(f"\n{'-'*80}")
            print(f"Question {i}/{len(quiz)} (ID: {question_id})")
            print(f"{'-'*80}")
            print(f"{question[:100]}...")
            
            try:
                # Run all three approaches
                zero_shot_result = self.run_zero_shot(gpqa_question)
                baseline_result = self.run_baseline(gpqa_question, baseline_graph)
                adaptive_result = self.run_adaptive(gpqa_question, adaptive_graph, max_iters)
                
                # Extract scores
                def get_overall_score(result):
                    return result.get("quiz_results", {}).get("overall_score", 0.0)
                
                question_results.append({
                    "question_id": question_id,
                    "question": question,
                    "gpqa_index": indices[i-1] if i-1 < len(indices) else None,
                    "correct_answer": gpqa_question["correct"],
                    "expert_explanation": gpqa_question.get("expert_explanation", ""),
                    "zero_shot": {
                        "quiz_performance": zero_shot_result["quiz_results"],
                        "overall_score": zero_shot_result["overall_score"],
                        "predicted": zero_shot_result["quiz_results"].get("predicted", "?")
                    },
                    "baseline": {
                        "quiz_performance": baseline_result["quiz_results"],
                        "overall_score": baseline_result["overall_score"],
                        "predicted": baseline_result.get("single_answer", "?"),
                        "explanation": baseline_result["explanation"],
                        "single_explanation": baseline_result.get("single_explanation", "")
                    },
                    "adaptive": {
                        "quiz_performance": adaptive_result["quiz_results"],
                        "overall_score": adaptive_result["overall_score"],
                        "predicted": adaptive_result.get("single_answer", "?"),
                        "explanation": adaptive_result["explanation"],
                        "single_explanation": adaptive_result.get("single_explanation", ""),
                        "iterations": adaptive_result.get("iterations", 0),
                        "final_scores": adaptive_result.get("final_scores", {})
                    }
                })
                
                print(f"\n  Results:")
                print(f"    Zero-shot: {'✓' if zero_shot_result['overall_score'] == 1.0 else '✗'} (answer: {zero_shot_result['quiz_results'].get('predicted', '?')})")
                print(f"    Baseline:  {'✓' if baseline_result['overall_score'] == 1.0 else '✗'} (answer: {baseline_result.get('single_answer', '?')})")
                print(f"    Adaptive:  {'✓' if adaptive_result['overall_score'] == 1.0 else '✗'} (answer: {adaptive_result.get('single_answer', '?')}, {adaptive_result.get('iterations', 0)} iterations)")
                
                
            except Exception as e:
                print(f"  ERROR: {e}")
                import traceback
                traceback.print_exc()
                question_results.append({
                    "question_id": question_id,
                    "question": question,
                    "error": str(e)
                })
        
        # Aggregate statistics
        summary = self._compute_summary(question_results)
        
        # Save results
        results = {
            "timestamp": timestamp,
            "config": {
                "subset": self.subset,
                "domain": self.domain,
                "n_questions": self.n_questions,
                "seed": self.seed,
                "max_iters": max_iters
            },
            "quiz": quiz,
            "question_results": question_results,
            "summary": summary
        }
        
        output_file = self.results_dir / f"eval_{timestamp}.json"
        output_file.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")
        
        # Print summary
        self._print_summary(summary)
        print(f"\n{'='*80}")
        print(f"Results saved to: {output_file}")
        print(f"{'='*80}\n")
        
        return results
    
    def _compute_summary(self, question_results: List[Dict[str, Any]]) -> Dict[str, Any]:
        """Compute aggregate statistics across all questions."""
        valid_results = [r for r in question_results if "error" not in r]
        n_valid = len(valid_results)
        
        if n_valid == 0:
            return {"error": "No valid results"}
        
        # Count correct answers
        zero_shot_correct = sum(1 for r in valid_results if r["zero_shot"]["overall_score"] == 1.0)
        baseline_correct = sum(1 for r in valid_results if r["baseline"]["overall_score"] == 1.0)
        adaptive_correct = sum(1 for r in valid_results if r["adaptive"]["overall_score"] == 1.0)
        
        # Count wins (who got it correct when others didn't)
        zero_shot_wins = sum(1 for r in valid_results 
                             if r["zero_shot"]["overall_score"] == 1.0
                             and (r["baseline"]["overall_score"] == 0.0 or r["adaptive"]["overall_score"] == 0.0))
        baseline_wins = sum(1 for r in valid_results 
                           if r["baseline"]["overall_score"] == 1.0
                           and (r["zero_shot"]["overall_score"] == 0.0 or r["adaptive"]["overall_score"] == 0.0))
        adaptive_wins = sum(1 for r in valid_results 
                           if r["adaptive"]["overall_score"] == 1.0
                           and (r["zero_shot"]["overall_score"] == 0.0 or r["baseline"]["overall_score"] == 0.0))
        
        # Average iterations for adaptive
        avg_iterations = sum(r["adaptive"]["iterations"] for r in valid_results) / n_valid
        
        return {
            "n_questions": n_valid,
            "accuracy": {
                "zero_shot": zero_shot_correct / n_valid,
                "baseline": baseline_correct / n_valid,
                "adaptive": adaptive_correct / n_valid
            },
            "correct_counts": {
                "zero_shot": zero_shot_correct,
                "baseline": baseline_correct,
                "adaptive": adaptive_correct
            },
            "wins": {
                "zero_shot": zero_shot_wins,
                "baseline": baseline_wins,
                "adaptive": adaptive_wins
            },
            "adaptive_metrics": {
                "average_iterations": avg_iterations
            },
            "improvements": {
                "adaptive_vs_zero_shot": (adaptive_correct - zero_shot_correct) / n_valid,
                "adaptive_vs_baseline": (adaptive_correct - baseline_correct) / n_valid,
                "baseline_vs_zero_shot": (baseline_correct - zero_shot_correct) / n_valid
            }
        }
    
    def _print_summary(self, summary: Dict[str, Any]):
        """Print formatted summary statistics."""
        print(f"\n{'='*80}")
        print("EVALUATION SUMMARY")
        print(f"{'='*80}")
        print(f"Valid Questions: {summary['n_questions']}")
        
        print(f"\nAccuracy (Correct Answers):")
        print(f"  Zero-shot: {summary['correct_counts']['zero_shot']}/{summary['n_questions']} ({summary['accuracy']['zero_shot']*100:.1f}%)")
        print(f"  Baseline:  {summary['correct_counts']['baseline']}/{summary['n_questions']} ({summary['accuracy']['baseline']*100:.1f}%)")
        print(f"  Adaptive:  {summary['correct_counts']['adaptive']}/{summary['n_questions']} ({summary['accuracy']['adaptive']*100:.1f}%)")
        
        print(f"\nWins (correct when at least one other was wrong):")
        print(f"  Zero-shot: {summary['wins']['zero_shot']}")
        print(f"  Baseline:  {summary['wins']['baseline']}")
        print(f"  Adaptive:  {summary['wins']['adaptive']}")
        
        print(f"\nImprovements (accuracy difference):")
        print(f"  Adaptive vs Zero-shot: {summary['improvements']['adaptive_vs_zero_shot']*100:+.1f}%")
        print(f"  Adaptive vs Baseline:  {summary['improvements']['adaptive_vs_baseline']*100:+.1f}%")
        print(f"  Baseline vs Zero-shot: {summary['improvements']['baseline_vs_zero_shot']*100:+.1f}%")
        
        print(f"\nAdaptive System:")
        print(f"  Average Iterations: {summary['adaptive_metrics']['average_iterations']:.1f}")

In [10]:
def extract_explanations_for_pairwise_judge(eval_results_file: Path) -> Dict[str, Any]:
    """
    Extract baseline and adaptive explanations from evaluation results
    and run pairwise judge comparison.
    
    Args:
        eval_results_file: Path to evaluation JSON file
        
    Returns:
        Dictionary with comparison results from pairwise judge
    """
    with open(eval_results_file, encoding="utf-8") as f:
        results = json.load(f)
    
    question_results = results.get("question_results", [])
    
    # Extract data for pairwise comparison
    questions = []
    expert_explanations = []
    baseline_explanations = []
    adaptive_explanations = []
    metadata = []
    
    for qr in question_results:
        if "error" in qr:
            continue
        
        questions.append(qr["question"])
        expert_explanations.append(qr.get("expert_explanation", ""))
        baseline_explanations.append(qr["baseline"]["explanation"])
        adaptive_explanations.append(qr["adaptive"]["explanation"])
        
        metadata.append({
            "question_id": qr["question_id"],
            "baseline_score": qr["baseline"]["overall_score"],
            "adaptive_score": qr["adaptive"]["overall_score"],
            "baseline_correct": qr["baseline"]["overall_score"] == 1.0,
            "adaptive_correct": qr["adaptive"]["overall_score"] == 1.0,
            "correct_answer": qr["correct_answer"],
            "baseline_predicted": qr["baseline"]["predicted"],
            "adaptive_predicted": qr["adaptive"]["predicted"]
        })
    
    if len(questions) == 0:
        return {
            "source_file": str(eval_results_file),
            "error": "No valid question results found",
            "summary": {}
        }
    
    print(f"\nRunning pairwise judge on {len(questions)} explanations...")
    print(f"Comparing: adaptive vs baseline\n")
    
    # Run batch comparison
    judge_results = batch_pairwise_comparison(
        questions=questions,
        expert_explanations=expert_explanations,
        explanations_a=adaptive_explanations,
        explanations_b=baseline_explanations,
        label_a="adaptive",
        label_b="baseline"
    )
    
    # Add metadata to individual results
    for i, meta in enumerate(metadata):
        if i < len(judge_results["individual_results"]):
            judge_results["individual_results"][i]["metadata"] = meta
    
    # Print summary
    summary = judge_results["summary"]
    print(f"{'='*80}")
    print("PAIRWISE JUDGE RESULTS")
    print(f"{'='*80}")
    print(f"Total Comparisons: {summary['total_comparisons']}")
    print(f"\nOverall Winners:")
    print(f"  Adaptive: {summary['adaptive_wins']} ({summary['adaptive_win_rate']:.1%})")
    print(f"  Baseline: {summary['baseline_wins']} ({summary['baseline_win_rate']:.1%})")
    print(f"  Ties:     {summary['ties']}")
    
    print(f"\nCriterion Breakdown:")
    for criterion in ["physics_correctness", "pedagogical_quality", "clarity_precision"]:
        adaptive_count = summary['criterion_breakdown']['adaptive'][criterion]
        baseline_count = summary['criterion_breakdown']['baseline'][criterion]
        total = summary['total_comparisons']
        ties_count = total - adaptive_count - baseline_count
        
        print(f"  {criterion.replace('_', ' ').title()}:")
        print(f"    Adaptive: {adaptive_count}, Baseline: {baseline_count}, Ties: {ties_count}")
    
    # Save results
    judge_output = eval_results_file.parent / f"judge_{eval_results_file.stem}.json"
    judge_output.write_text(json.dumps(judge_results, ensure_ascii=False, indent=2), 
                           encoding="utf-8")
    print(f"\n{'='*80}")
    print(f"Results saved to: {judge_output}")
    print(f"{'='*80}\n")
    
    return judge_results

In [6]:
# Create graphs once
baseline_graph = create_baseline_graph()
adaptive_graph = create_adaptive_refinement_graph()

# Run evaluation
pipeline = EvaluationPipeline(
    subset="gpqa_main",
    domain="Physics",
    n_questions=5,
    seed=32,
    results_dir = project_root / "results"
)

results = pipeline.run_full_evaluation(
    baseline_graph=baseline_graph,
    adaptive_graph=adaptive_graph,
    max_iters=3
)


Starting Evaluation Run: 20251118_150604
Dataset: gpqa_main - Physics
Questions: 5
Seed: 32

Generating quiz...
Loading GPQA cache from: C:\Users\vedan\Desktop\playing-devils-advocate\data\cache\gpqa_main_Physics_train.json
Loaded 182 Physics questions from gpqa_main
Generated 5 questions


--------------------------------------------------------------------------------
Question 1/5 (ID: reckbjNKYmpQlU5TY)
--------------------------------------------------------------------------------
A photon h\nu propagates in the Oz direction of the laboratory frame (R). It collides elastically wi...
  Running zero-shot...
  Running baseline...
  Running adaptive (max 3 iterations)...

  Results:
    Zero-shot: ✓ (answer: B)
    Baseline:  ✓ (answer: B)
    Adaptive:  ✓ (answer: B, 3 iterations)

--------------------------------------------------------------------------------
Question 2/5 (ID: recywRj5a8EEjj2Ib)
--------------------------------------------------------------------------------
Consi

In [11]:
results_file = Path(r"C:\Users\vedan\Desktop\playing-devils-advocate\results\eval_20251118_150604.json")
judge_results = extract_explanations_for_pairwise_judge(results_file)


Running pairwise judge on 5 explanations...
Comparing: adaptive vs baseline

PAIRWISE JUDGE RESULTS
Total Comparisons: 5

Overall Winners:
  Adaptive: 3 (60.0%)
  Baseline: 2 (40.0%)
  Ties:     0

Criterion Breakdown:
  Physics Correctness:
    Adaptive: 1, Baseline: 2, Ties: 2
  Pedagogical Quality:
    Adaptive: 3, Baseline: 2, Ties: 0
  Clarity Precision:
    Adaptive: 2, Baseline: 3, Ties: 0

Results saved to: C:\Users\vedan\Desktop\playing-devils-advocate\results\judge_eval_20251118_150604.json



In [12]:
# Create graphs once
baseline_graph = create_baseline_graph()
adaptive_graph = create_adaptive_refinement_graph()

# Run evaluation
pipeline = EvaluationPipeline(
    subset="gpqa_main",
    domain="Physics",
    n_questions=5,
    seed=64,
    results_dir = project_root / "results"
)

results = pipeline.run_full_evaluation(
    baseline_graph=baseline_graph,
    adaptive_graph=adaptive_graph,
    max_iters=3
)


Starting Evaluation Run: 20251118_152019
Dataset: gpqa_main - Physics
Questions: 5
Seed: 64

Generating quiz...
Loading GPQA cache from: C:\Users\vedan\Desktop\playing-devils-advocate\data\cache\gpqa_main_Physics_train.json
Loaded 182 Physics questions from gpqa_main
Generated 5 questions


--------------------------------------------------------------------------------
Question 1/5 (ID: recYI852ugl1pFhAv)
--------------------------------------------------------------------------------
Consider we have 4 particles produced each 1e-5` in the atmosphere at 13000m from the ground level:
...
  Running zero-shot...
  Running baseline...
  Running adaptive (max 3 iterations)...

  Results:
    Zero-shot: ✗ (answer: A)
    Baseline:  ✗ (answer: A)
    Adaptive:  ✗ (answer: A, 3 iterations)

--------------------------------------------------------------------------------
Question 2/5 (ID: reciOUAwJv4qTlXxs)
--------------------------------------------------------------------------------
Suppo

In [13]:
results_file = Path(r"C:\Users\vedan\Desktop\playing-devils-advocate\results\eval_20251118_152019.json")
judge_results = extract_explanations_for_pairwise_judge(results_file)


Running pairwise judge on 5 explanations...
Comparing: adaptive vs baseline

PAIRWISE JUDGE RESULTS
Total Comparisons: 5

Overall Winners:
  Adaptive: 0 (0.0%)
  Baseline: 5 (100.0%)
  Ties:     0

Criterion Breakdown:
  Physics Correctness:
    Adaptive: 1, Baseline: 2, Ties: 2
  Pedagogical Quality:
    Adaptive: 0, Baseline: 5, Ties: 0
  Clarity Precision:
    Adaptive: 0, Baseline: 5, Ties: 0

Results saved to: C:\Users\vedan\Desktop\playing-devils-advocate\results\judge_eval_20251118_152019.json



In [14]:
# Create graphs once
baseline_graph = create_baseline_graph()
adaptive_graph = create_adaptive_refinement_graph()

# Run evaluation
pipeline = EvaluationPipeline(
    subset="gpqa_main",
    domain="Physics",
    n_questions=5,
    seed=74,
    results_dir = project_root / "results"
)

results = pipeline.run_full_evaluation(
    baseline_graph=baseline_graph,
    adaptive_graph=adaptive_graph,
    max_iters=3
)


Starting Evaluation Run: 20251118_153344
Dataset: gpqa_main - Physics
Questions: 5
Seed: 74

Generating quiz...
Loading GPQA cache from: C:\Users\vedan\Desktop\playing-devils-advocate\data\cache\gpqa_main_Physics_train.json
Loaded 182 Physics questions from gpqa_main
Generated 5 questions


--------------------------------------------------------------------------------
Question 1/5 (ID: recIIUynGGpsGEYuo)
--------------------------------------------------------------------------------
A light beam is propagating through a glass with index of refraction n. The glass is moving at const...
  Running zero-shot...
  Running baseline...
  Running adaptive (max 3 iterations)...

  Results:
    Zero-shot: ✓ (answer: B)
    Baseline:  ✗ (answer: A)
    Adaptive:  ✗ (answer: D, 3 iterations)

--------------------------------------------------------------------------------
Question 2/5 (ID: rec0VuKUjt1SZ7NYv)
--------------------------------------------------------------------------------
Consi

In [16]:
results_file = Path(r"C:\Users\vedan\Desktop\playing-devils-advocate\results\eval_20251118_153344.json")
judge_results = extract_explanations_for_pairwise_judge(results_file)


Running pairwise judge on 5 explanations...
Comparing: adaptive vs baseline

PAIRWISE JUDGE RESULTS
Total Comparisons: 5

Overall Winners:
  Adaptive: 0 (0.0%)
  Baseline: 5 (100.0%)
  Ties:     0

Criterion Breakdown:
  Physics Correctness:
    Adaptive: 0, Baseline: 5, Ties: 0
  Pedagogical Quality:
    Adaptive: 0, Baseline: 5, Ties: 0
  Clarity Precision:
    Adaptive: 0, Baseline: 5, Ties: 0

Results saved to: C:\Users\vedan\Desktop\playing-devils-advocate\results\judge_eval_20251118_153344.json



In [17]:
# Create graphs once
baseline_graph = create_baseline_graph()
adaptive_graph = create_adaptive_refinement_graph()

# Run evaluation
pipeline = EvaluationPipeline(
    subset="gpqa_main",
    domain="Physics",
    n_questions=5,
    seed=12,
    results_dir = project_root / "results"
)

results = pipeline.run_full_evaluation(
    baseline_graph=baseline_graph,
    adaptive_graph=adaptive_graph,
    max_iters=3
)


Starting Evaluation Run: 20251118_154435
Dataset: gpqa_main - Physics
Questions: 5
Seed: 12

Generating quiz...
Loading GPQA cache from: C:\Users\vedan\Desktop\playing-devils-advocate\data\cache\gpqa_main_Physics_train.json
Loaded 182 Physics questions from gpqa_main
Generated 5 questions


--------------------------------------------------------------------------------
Question 1/5 (ID: recYI852ugl1pFhAv)
--------------------------------------------------------------------------------
Consider we have 4 particles produced each 1e-5` in the atmosphere at 13000m from the ground level:
...
  Running zero-shot...
  Running baseline...
  Running adaptive (max 3 iterations)...

  Results:
    Zero-shot: ✗ (answer: D)
    Baseline:  ✗ (answer: D)
    Adaptive:  ✗ (answer: C, 3 iterations)

--------------------------------------------------------------------------------
Question 2/5 (ID: recoKZlD5PwM9zbk6)
--------------------------------------------------------------------------------
Durin

In [20]:
results_file = Path(r"C:\Users\vedan\Desktop\playing-devils-advocate\results\eval_20251118_154435.json")
judge_results = extract_explanations_for_pairwise_judge(results_file)


Running pairwise judge on 5 explanations...
Comparing: adaptive vs baseline

PAIRWISE JUDGE RESULTS
Total Comparisons: 5

Overall Winners:
  Adaptive: 2 (40.0%)
  Baseline: 3 (60.0%)
  Ties:     0

Criterion Breakdown:
  Physics Correctness:
    Adaptive: 1, Baseline: 4, Ties: 0
  Pedagogical Quality:
    Adaptive: 3, Baseline: 2, Ties: 0
  Clarity Precision:
    Adaptive: 2, Baseline: 3, Ties: 0

Results saved to: C:\Users\vedan\Desktop\playing-devils-advocate\results\judge_eval_20251118_154435.json

